In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load data
data=pd.read_csv("./dataset/smartcart_customers.csv")


In [ ]:
data.shape  # (2240, 22) 
data.isnull().sum() # Income 24 -> take the median income and fill then with median values


## Data Preprocessing Learning

#### 1 Handle Missing Values

In [ ]:
data['Income']= data['Income'].fillna(data['Income'].median())

#### 2 Feature Engineering 

In [ ]:
# 1- Year of Birth into Age
data['Age']=2026-data['Year_Birth']

In [ ]:
# 2- Customer joining Days 
data['Dt_Customer']= pd.to_datetime(data['Dt_Customer'],dayfirst=True)
refrence_date=data['Dt_Customer'].max()

# Create A New Feature Customer tenure  Days 
data['Customer_Tenure']= (refrence_date - data['Dt_Customer']).dt.days ## total number of days 


In [ ]:
data.columns

In [ ]:
#3  Total - Spending 
data['Total_Spending']=data['MntWines']+data['MntFruits']+ data['MntMeatProducts']+data['MntFishProducts']+data['MntSweetProducts']+data['MntGoldProds']

In [ ]:
# 4  Children
data['Total_Children']=data["Kidhome"]+data['Teenhome']

In [ ]:
# 5  Education
data['Education'].value_counts()

# Convert into Three Basic Catgories

# Undergraduate  , Graduate, PostGraduate
data['Education']=data['Education'].replace({
    "Basic":"Undergraduate",
    "2n Cycle":"Undergraduate",
    "Graduate":"Graduation",
    "Master":"PostGraduate",
    "PhD":"PostGraduate"   
})

# data['Education'].value_counts()

In [ ]:
# 6  Marital Status 
data['Marital_Status'].value_counts()

data['Living_With']=data['Marital_Status'].replace({
    "Married":"Partner",
    "Together":"Partner",
    "Single":"Alone",
    "Divorced":"Alone",
    "Widow":"Alone",
    "Alone":"Alone",
    "Absurd":"Alone",
    "YOLO":"Alone"
})

data['Living_With'].value_counts()


#### Drop UNnecessary Columns

In [ ]:
# I Remove Columns
# data= data.drop("ID",axis=1)   data.columns

cols=['Year_Birth','Marital_Status','Kidhome','Teenhome','Dt_Customer']
spending_cols=['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']

drop_cols=cols+spending_cols

data_cleaned= data.drop(columns=drop_cols)

In [ ]:
data_cleaned.to_csv('customer_data_updated.csv', index=False)

In [ ]:
data_cleaned.columns

## Handling With Outliers 

In [ ]:
numeric_columns=["Income","Recency","Response","Age","Total_Spending","Total_Children"]

#  Relative plots of some features - Pair Plots
sns.pairplot(data_cleaned[numeric_columns]) 

plt.savefig('01-pair_plots.png', dpi=300, bbox_inches='tight')


In [ ]:
# Remove the Outliers 
print("Data Size With outliers",len(data_cleaned))

data_cleaned= data_cleaned[data_cleaned['Age']<90] # remove the age outliers
data_cleaned= data_cleaned[data_cleaned['Income']<600_000] # remove the Income Outliers

print("Data Size WithOut outliers",len(data_cleaned))

## Heatmap

In [ ]:
corr= data_cleaned.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(
    corr,
    annot=True,
    annot_kws={"size":6},
    cmap="coolwarm"
    
)

plt.savefig('02-heatmaps.png', dpi=300, bbox_inches='tight')
plt.show()

# income vs total_spending corr -> 0.79
# income vs catelog purchase -> 0.69
# income vs store purchsed -> 0.63
#  # of wesbite viste s -.65
# spending catelog purchses 0.78
# store purchases 0.68

## Feature Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
data.head()

In [ ]:
ohe=OneHotEncoder()
# drop first in clustering we does not need 
cat_cols=["Education","Living_With"]

encoded_cols=ohe.fit_transform(data_cleaned[cat_cols])


In [ ]:
encoded_df=pd.DataFrame(
    encoded_cols.toarray(),
    columns=ohe.get_feature_names_out(cat_cols),
    index=data_cleaned.index
)

In [ ]:
encoded_data=pd.concat([data_cleaned.drop(columns=cat_cols), encoded_df],axis=1 )

## Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X=encoded_data
sc= StandardScaler()
X_scaled= sc.fit_transform(X)

## Visulize Data

In [ ]:
# Apply PCA TO REDUCE DIMENSIONS 
# 2D
from sklearn.decomposition import PCA

In [ ]:
pca=PCA(n_components=2)
X_pca= pca.fit_transform(X_scaled)

In [ ]:
# Plot
plt.scatter(X_pca[:,0],X_pca[:,1])
plt.xlabel("X PCA-1")
plt.ylabel('X-PCA-2')
plt.title("Scatter plot PCA-1  AND PCA-2")
plt.savefig('03-scatter-pca.png', dpi=300, bbox_inches='tight')
plt.show()
pca.explained_variance_ratio_  # array([0.23163158, 0.11385454])


In [ ]:
# 3D 
pca=PCA(n_components=3)
X_pca= pca.fit_transform(X_scaled)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
fig=plt.figure(figsize=(8,6))
ax=fig.add_subplot (111,projection='3d')
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2])
ax.set_xlabel("PCA-1")
ax.set_ylabel("PCA-2")
ax.set_zlabel("PCA-3")
ax.set_title("3d Projection ")
plt.savefig('04-3d_projectio.png', dpi=300, bbox_inches='tight')
plt.show()

## Analyze the value of k 

### 1-Using Elbow Method

In [ ]:
# By Using Elbow Method

from sklearn.cluster import KMeans
from kneed import KneeLocator

wcss=[]
for k in range(1,11):
    kmeans=KMeans(n_clusters=k,random_state=42)
    kmeans.fit_predict(X_pca)
    wcss.append(kmeans.inertia_)


knee=KneeLocator(range(1,11),wcss,curve='convex',direction='decreasing')

optimal_k=knee.elbow

print(f"Best k : {optimal_k}")

In [ ]:
#  plot for elbow method
plt.plot(range(1,11),wcss, marker='o')
plt.xlabel("K")
plt.ylabel("WCSS")
plt.title("Elbow Method plot for k ")
plt.savefig('05_elbow_method.png', dpi=300, bbox_inches='tight')


### 2- Silhouette Score 

In [ ]:
from sklearn.metrics import silhouette_score
scores=[]

for k in range(2,11):
    kmeans=KMeans(n_clusters=k,random_state=42)
    labels=kmeans.fit_predict(X_pca)
    score=silhouette_score(X_pca,labels)
    scores.append(score)

# PLot

#  plot for elbow method
plt.plot(range(2,11),scores, marker='o')
plt.xlabel("K")
plt.ylabel("Silhouetter Score")
plt.title("Silhouetter Score plot for k ")
plt.savefig('06_Silhouetter_Score.png', dpi=300, bbox_inches='tight')
plt.show()

     

In [ ]:
# Combined Plot for both Values of K

k_range=range(2,11)
fig,ax1=plt.subplots(figsize=(8,6))

ax1.plot(k_range,wcss[:len(k_range)],marker='o', color="blue")
ax1.set_xlabel("K")
ax1.set_ylabel("WCSS")

ax2=ax1.twinx() # same x axis
ax2.plot(k_range,scores[:len(k_range)],marker='x',color='orange',linestyle='--')
ax2.set_ylabel("Scores")
plt.title("Plots for  Value of K ")
plt.savefig('07_k_value.png', dpi=300, bbox_inches='tight')
plt.show()

## Clustering
### 1- KMeans

In [ ]:
kmeans= KMeans(n_clusters=4,random_state=42)
labels_kmeans=kmeans.fit_predict(X_pca)

In [ ]:
fig=plt.figure(figsize=(8,6))
ax=fig.add_subplot (111,projection='3d')
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2], c=labels_kmeans)
ax.set_title("Kmeans Algorithm  ")
plt.savefig('08-kmeans.png', dpi=300, bbox_inches='tight')
plt.show()

### 2 Agglomerative  Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering

In [ ]:
agg_clf=AgglomerativeClustering(n_clusters=4,linkage="ward")
lables_agg=agg_clf.fit_predict(X_pca)

fig=plt.figure(figsize=(8,6))
ax=fig.add_subplot (111,projection='3d')
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2], c=labels_kmeans)
ax.set_title("Agglomerative Algorithm  ")
plt.savefig('09-agglomerative.png', dpi=300, bbox_inches='tight')
plt.show()



## Characterization of Clusters

In [ ]:
data_cleaned.head()

In [ ]:
X['clusters']=lables_agg

In [ ]:
pal=['red','blue','yellow','green']
sns.countplot(x=X['clusters'], palette=pal,hue=X['clusters'])
plt.title("Labels for Clusters")
plt.savefig('10_cluster_out.png', dpi=300, bbox_inches='tight')
plt.show()


#### 1- Income vs Spending Patterns

In [ ]:
sns.scatterplot(x=X['Total_Spending'],y=X['Income'],hue=X['clusters'],palette=pal)

plt.title("Income Vs Total Sanding")
plt.savefig('11_incomevsspending_out.png', dpi=300, bbox_inches='tight')
plt.show()


### Customer Clustering Analysis

The customers are divided into **4 clusters** based on income and spending behavior:

- **Cluster 0:** Low–Moderate Income → Low–Moderate Spending
- **Cluster 1:** High Income → High Spending
- **Cluster 2:** Low Income → Low Spending
- **Cluster 3:** Moderate–High Income → High Spending

## Summary

- **Cluster 1 & 3:** High Income + High Spending
- **Cluster 0 & 2:** Low Income + Low Spending


## Cluster Summary

In [ ]:
cluster_summary=X.groupby("clusters").mean()
print(cluster_summary)

# Cluster Summar & Marketing Strategy

## 🔴 Cluster 0 — Family Shoppers
- More children
- Poor response rate
- Mostly partners
- High web visits
- Low web purchases
- Low store and catalog purchases

**Strategy:** Offer attractive **discounts and promotions** to increase engagement and purchases.

## 🔵 Cluster 1 — Loyalty Programs

- Fewer children
- Slightly higher age
- Average response rate
- Mostly partners
- Low web visits
- High store and catalog purchases
- High web purchases

**Strategy:** Introduce **loyalty programs and rewards** to retain these valuable customers.

## 🟡 Cluster 2 — Digital Browsers

- More children
- Average response rate
- Mostly alone
- High web visits
- Low web purchases
- Very low store and catalog purchases

**Strategy:** Target them with **sales, heavy discounts, and online promotions** to convert browsing into purchases.

## 🟢 Cluster 3 — Golden / Best ROI

- Fewer children
- Slightly higher age
- Better response rate
- Mostly alone
- Low web visits
- High store and catalog purchases
- High web purchases

**Strategy:** These are **high-value customers**. Offer **premium services, exclusive offers, and personalized experiences**.


## 📊 Overall Strategy

| Cluster | Segment | Recommended Strategy |
|---|---|---|
| 🔴 0 | Family Shoppers | Discounts & Promotions |
| 🔵 1 | Loyalty Programs | Loyalty & Rewards |
| 🟡 2 | Digital Browsers | Heavy Discounts & Online Sales |
| 🟢 3 | Golden / Best ROI | Premium Services & Exclusive Offers |